# Module 3 — Multilingual Feature Extraction (Google Colab)

**Explainable Multilingual Fake News Detection using XLM-RoBERTa, Supervised Contrastive Learning, Fact Verification, and Explainable AI for Low-Resource Languages**

## Runtime → Change runtime type → **T4 GPU**, then **Runtime → Run all**.

One execution produces everything Module 3 needs:

| Step | What happens |
|---|---|
| 1-3 | Mount Drive, locate project, install dependencies |
| 4 | Report GPU / CUDA / PyTorch environment |
| 5 | Confirm Module 2's finalized splits are present |
| 6 | **Extraction** — XLM-RoBERTa embeddings, auto-resuming from checkpoints |
| 7 | **Verification** — all 18 checks, runs automatically |
| 8 | Display the final verdict and reports |

Everything is written straight to Drive, so nothing is lost when the runtime
disconnects. If the session drops mid-extraction, just **Run all** again —
completed splits are skipped and partial splits resume from their last
checkpoint. Nothing is ever recomputed unnecessarily.

Module 1 and Module 2 are **not** re-run: this notebook only reads their
finalized outputs.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Locate the project folder

Edit `DRIVE_PROJECT_PATH` if the project lives somewhere other than
`My Drive/Fake_News_Detection`.

In [ ]:
import os
import sys
from pathlib import Path

DRIVE_PROJECT_PATH = "/content/drive/MyDrive/Fake_News_Detection"

PROJECT_ROOT = Path(DRIVE_PROJECT_PATH)
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project folder not found at {PROJECT_ROOT}.\n\n"
        "Module 3 reads Module 2's finalized splits from Drive, so the project "
        "must be synced there first. Copy the Fake_News_Detection folder to "
        "My Drive (datasets/processed/train.csv, validation.csv and test.csv are "
        "required; the large cleaned_dataset.* and module1_dataset.* files are "
        "not needed), or edit DRIVE_PROJECT_PATH above."
    )

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("Working directory:", os.getcwd())


## 3. Install dependencies

In [ ]:
!pip install -q transformers sentencepiece torch --upgrade
print("Dependencies installed.")


## 4. Execution environment

Recorded into `feature_statistics.json` by the extraction run itself, so the
final report carries the real hardware details rather than assumptions.

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"Total GPU memory: {props.total_memory / 1e9:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
    print("\nMixed precision will be ACTIVE (fp16 autocast).")
else:
    print("\n*** No GPU detected. ***")
    print("Extraction will still run correctly on CPU, but expect several hours")
    print("instead of ~10-20 minutes, and mixed precision will be inactive.")
    print("Runtime -> Change runtime type -> T4 GPU is strongly recommended.")


## 5. Confirm Module 2's finalized (Version B) splits are present

Module 3 reads these; it never regenerates or modifies them.

In [ ]:
import pandas as pd

processed_dir = PROJECT_ROOT / "datasets" / "processed"
required = ["train.csv", "validation.csv", "test.csv"]
missing = [f for f in required if not (processed_dir / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing Module 2 output file(s): {missing} in {processed_dir}.\n"
        "Sync Module 2's finalized outputs to Drive before running Module 3."
    )

total = 0
for f in required:
    df = pd.read_csv(processed_dir / f, low_memory=False)
    total += len(df)
    print(f"{f:16s} {len(df):>7,} rows   languages={df['language'].value_counts().to_dict()}")
print(f"{'TOTAL':16s} {total:>7,} rows")


## 6. Feature extraction

Loads real `xlm-roberta-base`, then for each split: tokenizes with dynamic
padding and truncation at 512 tokens, runs the frozen encoder forward pass
(mixed precision on GPU), extracts **both** CLS and mean-pooled embeddings,
L2-normalizes them, and saves NumPy + PyTorch arrays alongside
`record_id` / `label` / `language`.

**Automatic resume.** Checkpoints are written every 20 batches. A split whose
outputs already exist is skipped entirely; a partially-done split resumes from
its last checkpoint. Re-running this cell after a disconnect is always safe and
never recomputes finished work.

The run also records everything the final report needs — GPU model, CUDA and
PyTorch versions, peak GPU memory, per-batch timings, throughput, sequence-length
distribution, truncation counts, and checkpoint activity.

In [ ]:
!python scripts/extract_features.py


## 7. Verification — 18 checks, automatic

Runs immediately after extraction. Self-contained: if this Drive copy of the
project doesn't have `scripts/verify_module3_outputs.py`, the identical source
(base64-embedded below, so no quoting can corrupt it) is written out first.

Verifies embedding counts, split correspondence, missing/duplicate embeddings,
NaN/Inf, dimensions, label / language / record-ID alignment, cross-split data
leakage, Tamil Unicode integrity, checkpoint recovery, output-file presence,
storage sizes, runtime statistics, and GPU statistics — then reports a single
**PASS or FAIL with reasons**.

This only reads what extraction produced; it never regenerates embeddings.

In [ ]:
import base64
import subprocess
import sys
from pathlib import Path

VERIFIER_PATH = Path("scripts/verify_module3_outputs.py")

_VERIFIER_B64 = (
    "..."
)

if not VERIFIER_PATH.exists():
    VERIFIER_PATH.parent.mkdir(parents=True, exist_ok=True)
    VERIFIER_PATH.write_text(base64.b64decode(_VERIFIER_B64).decode("utf-8"), encoding="utf-8")
    print(f"Wrote {VERIFIER_PATH} (was not present in this Drive copy).")
else:
    print(f"Using existing {VERIFIER_PATH}.")

result = subprocess.run([sys.executable, str(VERIFIER_PATH)], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("--- stderr ---")
    print(result.stderr)
print(f"exit code: {result.returncode}  (0 = verified successfully)")


## 8. Final result

Prints the verdict and the four generated reports, so the outcome is captured
in this notebook's saved output as well as in the files on Drive.

In [ ]:
import json
from pathlib import Path

reports_dir = PROJECT_ROOT / "outputs" / "module3"
verification_json = reports_dir / "final_verification_report.json"

if verification_json.exists():
    payload = json.loads(verification_json.read_text(encoding="utf-8"))
    print("=" * 72)
    print(payload["verdict"])
    print("=" * 72)
    print(f"{payload['n_passed']} passed / {payload['n_failed']} failed / "
          f"{payload['n_unavailable']} unavailable  ({payload['n_checks']} checks)")
    if payload.get("failure_reasons"):
        print("\nReasons:")
        for reason in payload["failure_reasons"]:
            print(f"  - {reason}")
    print()

    for check in payload["checks"]:
        print(f"  [{check['status']:^11s}] {check['check']:2d}. {check['name']}")
else:
    print(f"{verification_json} not found - did step 7 run successfully?")

print("\n" + "=" * 72)
print("GENERATED FILES")
print("=" * 72)
for name in ["feature_statistics.json", "feature_report.md",
             "final_verification_report.json", "final_verification_report.md",
             "feature_extraction.log"]:
    p = reports_dir / name
    print(f"  {'OK ' if p.exists() else 'MISSING'}  outputs/module3/{name}"
          + (f"  ({p.stat().st_size:,} bytes)" if p.exists() else ""))

embeddings_dir = PROJECT_ROOT / "embeddings"
if embeddings_dir.exists():
    total = sum(f.stat().st_size for f in embeddings_dir.rglob("*") if f.is_file())
    print(f"\n  embeddings/  total {total / 1024**2:,.1f} MB")
    for split in ["train", "validation", "test"]:
        d = embeddings_dir / split
        if d.exists():
            n = sum(1 for f in d.iterdir() if f.is_file())
            size = sum(f.stat().st_size for f in d.iterdir() if f.is_file())
            print(f"    {split:12s} {n} files, {size / 1024**2:,.1f} MB")


## 9. Read the full report

In [ ]:
report = PROJECT_ROOT / "outputs" / "module3" / "final_verification_report.md"
print(report.read_text(encoding="utf-8") if report.exists() else f"{report} not found.")


## Done

Module 3 is complete when step 8 prints **MODULE 3 VERIFIED SUCCESSFULLY**.

Everything is already saved to Drive under the project folder:

- `embeddings/{train,validation,test}/{cls,mean}_embeddings.{npy,pt}` plus `record_ids` / `labels` / `languages`
- `outputs/module3/feature_statistics.json` and `feature_report.md` — real measured values
- `outputs/module3/final_verification_report.json` and `.md` — the 18-check verdict
- `outputs/module3/feature_extraction.log`

Send those four `outputs/module3/` reports back for final review. Module 4
(Supervised Contrastive Learning) does not begin until Module 3 is explicitly
approved.